# 17 ECBS / 有界次优 CBS


本课按百科条目写。实时系统常要「保证不太差」而不是「保证最优」。调度台在最后。

---

## 1. 定义

**ECBS**（Barer, Sharon, Stern, Felner, SoCS 2014）允许

\[
SOC \le w\cdot OPT,\qquad w\ge 1
\]

高层、底层都用 **focal search**：在 f ≤ w·当前最优下界 的节点里，优先扩「冲突少」的，而不是严格最小 f。

**EECBS**（Li et al., AAAI 2021）：高层换 Explicit Estimation Search，界限同类。

本仓库教学版：先 CBS 得 OPT，若 HCA 种子落在 w·OPT 内就用种子，否则退回 CBS。用来理解 **界限**，不是完整 focal。请诚实对待：运行时你事先不知道 OPT；真 ECBS 用各车 focal 下界之和当 LB。

---

## 2. 和 CBS / 加权 A*

| | 加权 A\* | ECBS |
|--|--|--|
| 对象 | 单车路径 | 多车 SOC |
| 键 | g+w h | 高层 focal + 底层 focal |
| w=1 | A\* 最优 | 必须回到 CBS 最优 |

「超时返回当前最好」没有 w·OPT 证明。有界次优的价值是 **证明**，不是感觉。

---

## 3. 不变量

- w=1 ⇒ 与 CBS 同解（教学版用 CBS 本身）。
- 返回的 SOC ≤ w·OPT（教学版：种子通过才用种子，否则 OPT）。
- 真 ECBS 的 LB 是可采纳的，不依赖先算出 OPT。

---

## 4. 伪代码

完整 ECBS 概念：

```
底层：focal A*，在 g+h ≤ w·lb 的点里扩冲突最少的
高层：focal 于 SOC 下界 ≤ w·全局下界 的 CT 节点
第一个可行解即 w-次优
```

本课教学：

```
opt ← CBS(...)
seed ← HCA(...)
if seed.soc ≤ w * opt.soc: return seed
else: return opt
```

---

## 5. 手算

OPT=12，HCA 种子=16。

- w=1：16 ≰ 12，返回 CBS。
- w=1.5：上限 18，16≤18，可用种子。
- w=3：同样可用种子。


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "lib").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))


## 6. 实现：界限检查


In [ ]:
from evals.scenarios import two_agent_swap
from lib.algos.cbs import cbs
from lib.algos.prioritized import hca


def ecbs_teach(graph, starts, goals, w=1.5):
    opt = cbs(graph, starts, goals)
    seed = hca(graph, starts, goals)
    bound = w * opt["soc"]
    print("OPT", opt["soc"], "HCA种子", seed.get("soc"), "上限", bound)
    if seed["found"] and seed["soc"] <= bound + 1e-9:
        seed["w"] = w
        seed["bound"] = bound
        seed["optimal_soc"] = opt["soc"]
        seed["used"] = "hca"
        return seed
    opt["w"] = w
    opt["bound"] = bound
    opt["optimal_soc"] = opt["soc"]
    opt["used"] = "cbs"
    return opt


g, starts, goals = two_agent_swap()
for w in (1.0, 1.5, 3.0):
    r = ecbs_teach(g, starts, goals, w=w)
    print(f"w={w} used={r['used']} SOC={r['soc']} 上限={r['bound']}")


In [ ]:
from lib.algos.cbs import ecbs, cbs

opt = cbs(g, starts, goals)
for w in (1.0, 1.5):
    r = ecbs(g, starts, goals, w=w)
    print(f"库 w={w} SOC={r['soc']} 上限={w * opt['soc']:.1f} 应成立 {r['soc'] <= w * opt['soc'] + 1e-9}")


## 常见 bug

1. 用真实 OPT 当运行时下界（你事先不知道 OPT）。真正 ECBS 用各车 fmin 之和。
2. w=1 却用了 inadmissible 底层。
3. 把 HCA 种子直接当 ECBS，不检查界限。
4. 和 anytime CBS（先出可行再改善）搞混：anytime 无 w 证明。

## 条目小结

| 项目 | 内容 |
|------|------|
| 保证 | SOC ≤ w·OPT |
| w=1 | 最优 |
| 教学版 | CBS + 可选 HCA 种子 |
| 下一课 | 已有可行解再抠质量：LNS |

## 练习

1. 为何实时系统更爱「w-次优」而不是「超时返回当前最好」？
2. 若 HCA 种子比 w·OPT 还差，教学版退回 CBS。真 ECBS 会怎样？
3. 单车加权 A\* 的 w 和 ECBS 的 w 能直接比较吗？一个管路径，一个管 SOC。


In [ ]:
from lib.studio.widget import PlannerStudio
from lib.algos.cbs import ecbs
r = ecbs(g, starts, goals, w=1.5)
studio = PlannerStudio()
studio.show_multi(g.to_studio(), r, "ECBS")
studio
